# V2 push, step 2 — SFT v2 on data v1 (tracer, seed 3407)

**Runtime: A100 (~35 min) or L4 (~1h10).** Parallel-safe with notebook 11's L4
session if you use the A100 here.

Same proven recipe as notebook 04 (QLoRA r32/α64, loss on answers, 1-epoch-vs-
2-epoch verdict by dev pass@16) — the ONLY changes, all data-side:
- **data v1**: 2,076 train bugs (544 v0 + 1,532 DeepSeek self-broken, docstring
  format, subtle + compound tiers) — 3.8× the v0 train set
- restraint = the 303 docstringed train-split clean functions (~13% of mix)
- no routing upweighting (v0's sft-pile×3 was tuned for the 672-bug corpus;
  v2.0 starts plain — routing v1 comes later if needed)
- max_seq_length 1280 (docstrings lengthen examples; avoids truncation)

**Evals for comparability:** the v0 dev slice (61 bugs, k=16) is the ruler —
directly comparable to every previous number (v1-SFT s3407 = 58.7/91.8).
A second eval on 60 NEW dev bugs (k=8) shows performance on the harder
docstring-style distribution. Restraint probe after each epoch.

In [ ]:
# Setup: Drive, repo, data v1 (Drive phase8) + v0 dev slice (repo, the ruler)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
v1_bugs = json.load(open(f'{PHASE8}/data_v1_bugs.json'))
v1_restraint = json.load(open(f'{PHASE8}/data_v1_restraint.json'))
v0_bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
v0_restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
train_v1 = [b for b in v1_bugs if b['split'] == 'train']
dev_v0 = [b for b in v0_bugs if b['split'] == 'dev']
dev_new = [b for b in v1_bugs if b['split'] == 'dev' and b.get('gen') == 'deepseek_v1']
dev_clean = [r for r in v0_restraint if r['split'] == 'dev']
clean_train = [r for r in v1_restraint if r['split'] == 'train']
SEED = 3407
print(len(train_v1), 'v1 train bugs |', len(dev_v0), 'v0 dev (ruler) |',
      len(dev_new), 'new dev |', len(clean_train), 'restraint train')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Mixture: all v1 train bugs x1 + all 303 restraint (~13%)
rng = random.Random(SEED)
mixture = [{'prompt': build_repair_prompt(b['buggy'], b['test_list']),
            'answer': '```python\n' + b['fixed'].strip() + '\n```'} for b in train_v1]
mixture += [{'prompt': build_repair_prompt(r['code'], r['test_list']),
             'answer': '```python\n' + r['code'].strip() + '\n```'} for r in clean_train]
rng.shuffle(mixture)
print(f'{len(mixture)} examples ({len(clean_train)} restraint = {len(clean_train)/len(mixture)*100:.0f}%)')

In [ ]:
# Model: base Qwen 4-bit + LoRA (r32/α64, seed 3407), ctx 1280 for docstrings
from unsloth import FastLanguageModel
import torch
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1280,
    load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)
from datasets import Dataset
def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['answer']}]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}
ds = Dataset.from_list(mixture).map(to_text)
print('ONE FORMATTED EXAMPLE (docstring should be visible in the buggy code):')
print(ds[0]['text'][:1200])

In [ ]:
# Eval helpers: v0-dev ruler (k=16), new-dev view (k=8), restraint probe
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

@torch.no_grad()
def sample_k(source_code, test_list, k):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': build_repair_prompt(source_code, test_list)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def eval_slice(bugs, tag, k):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in bugs:
        codes = sample_k(b['buggy'], b['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    pk = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, f'pass@{k}': pk, 'gap': pk - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE8}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@{k}={pk*100:.1f}%  gap={(pk-p1)*100:.1f}")
    return res

def restraint_probe(tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: ' '.join(s.split())
    sp = un = tot = 0
    for r in dev_clean:
        codes = sample_k(r['code'], r['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes(c, r), codes))
        sp += sum(flags); un += sum(norm(c) == norm(r['code']) for c in codes); tot += k
    print(f"[probe {tag}]  still passes: {sp/tot*100:.1f}%  unchanged: {un/tot*100:.1f}%")

dev_new_sample = random.Random(SEED).sample(dev_new, 60)
print('helpers ready; new-dev sample =', len(dev_new_sample))

In [ ]:
# TRAIN ep1 -> evals -> TRAIN ep2 -> evals (same trainer recipe as notebook 04)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def make_trainer():
    FastLanguageModel.for_training(model)
    t = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
        dataset_text_field='text',
        args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
            num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_ratio=0.05, logging_steps=10, seed=SEED, output_dir='/content/out',
            report_to='none', save_strategy='no'))
    return train_on_responses_only(t,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

make_trainer().train()
model.save_pretrained(f'{PHASE8}/sft_v2_s{SEED}_ep1')
ep1 = eval_slice(dev_v0, f'sftv2_ep1_seed{SEED}', 16)
eval_slice(dev_new_sample, f'sftv2_ep1_newdev_seed{SEED}', 8)
restraint_probe(f'sftv2_ep1')

make_trainer().train()
model.save_pretrained(f'{PHASE8}/sft_v2_s{SEED}_ep2')
ep2 = eval_slice(dev_v0, f'sftv2_ep2_seed{SEED}', 16)
eval_slice(dev_new_sample, f'sftv2_ep2_newdev_seed{SEED}', 8)
restraint_probe(f'sftv2_ep2')

In [ ]:
# VERDICT vs the v1-generation SFT (same seed, same v0-dev ruler)
print(f"{'model (s3407, v0-dev)':<26} pass@1   pass@16   gap")
print(f"{'base':<26}   45.9%    93.4%   47.5")
print(f"{'SFT v1 (672 bugs)':<26}   58.7%    91.8%   33.1")
for name, r in (('SFT v2 ep1 (2076 bugs)', ep1), ('SFT v2 ep2', ep2)):
    print(f"{name:<26} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
print('\nEpoch rule (A2): pick by pass@16, never by pass@1 alone.')
print('Ruler: ~1.5 pts = tie. If v2 beats v1-SFT clearly, the data lever worked;')
print('then GRPO v2 (+ its own random-reward twin) trains on top of this.')
print('Bring the whole printout back.')

## Bring back to the session
1. The **formatted-example printout** (docstring visible in the buggy code?)
2. **Both epochs' three eval lines** (v0-dev ruler, new-dev view, restraint probe)
3. The **verdict table**

After this: GRPO v2 on the winning checkpoint — re-gated pile from data v1,
edit-aware penalty, 500 steps, WITH a matched random-reward twin for attribution.